# 🚀 Running this Notebook in Google Colab

### Prerequisites & Kernel Setup:
This notebook contains **Scala** code. To run it smoothly in Google Colab:
1. **Method 1 (Native Scala via Almond Kernel - Recommended):**
   - Run the setup cell below. It installs the **Almond Scala kernel** into your Colab VM in ~30 seconds.
   - Once it completes, click **Runtime** ➔ **Manage Sessions** ➔ terminate the active session (or reload your browser tab).
   - Reconnect to the notebook and all Scala cells will execute natively!
2. **Method 2 (Zero-Setup Companion Script):**
   - If you prefer not to switch kernels, each exercise also has a standalone `.scala` file included in this directory. You can run it via the bash execution cell at the end of the notebook!


In [ ]:
# [Colab Setup Cell] Run this to install Almond Scala Kernel in Google Colab
# (Only needed once per session)
!curl -Lo coursier https://git.io/coursier-cli && chmod +x coursier
!./coursier launch --fork almond:0.13.14 --scala 2.13.12 -- --install
println("Almond Scala Kernel installed successfully! Reload tab if prompted.")


# Exercise 18: Data Manipulation with Saddle & Interactive Visualization with Vegas

## Problem Statement:
1. Use **Saddle** to load and manipulate a dataset of fruit prices and quantities over time.
2. Filter the dataset for **apples** and calculate their **average price**.
3. Visualize the price trend over time using **Vegas**.

---

## Theoretical Deep Dive: Saddle & Vegas
- **Saddle (`org.saddle`)**:
  - Saddle is the high-performance data library for Scala, bringing the indexed, vector-backed architecture of Python's **Pandas** to the JVM.
  - Core structures:
    - `Vec[T]`: Primitive array wrapper enabling vectorized calculations and missing-data handling.
    - `Series[K, V]`: Indexed 1D vector.
    - `Frame[RowKey, ColKey, Val]`: 2D indexed tabular matrix.
- **Vegas (`vegas._`)**:
  - Declarative statistical visualization library wrapping **Vega-Lite**.
  - Maps data dimensions to visual encoding channels (`.encodeX`, `.encodeY`) and marks (`Line`, `Bar`, `Point`).


In [ ]:
// Ivy imports for Almond kernel in Colab
import $ivy.`org.saddle::saddle-core:1.3.4`
import $ivy.`org.vegas-viz::vegas:0.3.11`
import $ivy.`org.vegas-viz::vegas-spark:0.3.11`

println("Saddle and Vegas libraries successfully imported!")


In [ ]:
import org.saddle._

// 1. Create vectors for Date, Fruit, Price, and Quantity
val dates      = Vec("2023-10-01", "2023-10-02", "2023-10-03", "2023-10-04", "2023-10-05", "2023-10-01", "2023-10-02")
val fruitNames = Vec("Apple", "Apple", "Apple", "Apple", "Apple", "Banana", "Orange")
val prices     = Vec(2.40, 2.65, 2.50, 2.90, 3.10, 1.20, 1.80)
val quantities = Vec(120.0, 110.0, 140.0, 95.0, 130.0, 250.0, 180.0)

// Construct a Saddle 2D Frame
val fruitFrame: Frame[Int, String, Any] = Frame(
  "Date"     -> dates,
  "Fruit"    -> fruitNames,
  "Price"    -> prices,
  "Quantity" -> quantities
)

println("--- Full Fruit Dataset (Saddle Frame) ---")
println(fruitFrame)


In [ ]:
// 2. Filter for apples and calculate their average price
val fruitVec = fruitFrame.col("Fruit").toVec
val isAppleMask = fruitVec.map(_ == "Apple")

// Filter rows where fruit is Apple
val appleFrame = fruitFrame.filterRows(r => isAppleMask.at(r))
println("--- Filtered Dataset: Apples Only ---")
println(appleFrame)

// Extract Apple prices as Double Vec and compute mean
val applePrices = appleFrame.col("Price").mapValues(_.asInstanceOf[Double]).toVec
val avgApplePrice = applePrices.mean

println(f"Number of Apple Observations: ${applePrices.length}")
println(f"Average Apple Price: $$${avgApplePrice}%.2f")


In [ ]:
import vegas._
import vegas.render.HTML5._

// 3. Prepare data and visualize price trend over time using Vegas
val applePlotData = Seq(
  Map("Date" -> "2023-10-01", "Price" -> 2.40),
  Map("Date" -> "2023-10-02", "Price" -> 2.65),
  Map("Date" -> "2023-10-03", "Price" -> 2.50),
  Map("Date" -> "2023-10-04", "Price" -> 2.90),
  Map("Date" -> "2023-10-05", "Price" -> 3.10)
)

val plot = Vegas("Apple Price Trend Over Time")
  .withData(applePlotData)
  .encodeX("Date", Nom)
  .encodeY("Price", Quant)
  .mark(Line)

println("--- Generated Vega-Lite Visualization Specification ---")
println(plot.asVega_Lite)

// In Jupyter/Almond, calling plot.show renders the interactive visualization inline!
